# GeoClip Fine-Tuned Evaluation

Evaluate the GeoClip model **after fine-tuning** on the MMlandmarks train split.
The checkpoint is produced by `scripts/geoclip_train.py` and saved to
`models/best_geoclip_baseline.pth` (best Acc@25km across training epochs).

**Gallery:** 17,557 train landmark GPS coordinates (closed-world).
**Queries:** same query landmarks used in the zero-shot notebook.
**Metric:** Accuracy @ {1, 25, 200, 750, 2500} km (Haversine distance).

Compare these numbers directly to `03_geoclip_zeroshot.ipynb` to see the lift from fine-tuning.

## 1. Setup

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml

plt.rcParams.update({"figure.dpi": 120})

with open("../../configs/geoclip_baseline.yaml") as f:
    cfg = yaml.safe_load(f)

DATA_ROOT = Path("../../") / cfg["data"]["root"]
assert DATA_ROOT.exists(), f"DATA_ROOT not found: {DATA_ROOT}"

CHECKPOINT_PATH = Path("../../models/best_geoclip_baseline.pth")
assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}. Run scripts/geoclip_train.py first."
)

device = cfg["inference"]["device"] if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Checkpoint: {CHECKPOINT_PATH.resolve()}")

## 2. Load Model + Fine-Tuned Checkpoint

In [ ]:
from mmgeo.geolocalizations.geoclip.geoclip_baseline import (
    GeoClipBaseline,
    load_gallery_coords,
    load_query_data,
)
from mmgeo.geolocalizations.geoclip.evaluate import (
    accuracy_at_thresholds,
    median_error,
    haversine,
)

baseline = GeoClipBaseline(device=device)
baseline.model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
baseline.model.eval()

total_params = sum(p.numel() for p in baseline.model.parameters())
trainable_params = sum(p.numel() for p in baseline.model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("Loaded fine-tuned checkpoint.")

## 3. Build GPS Gallery

In [ ]:
gallery_coords = load_gallery_coords(
    DATA_ROOT, include_index=cfg["gallery"]["include_index"]
)
print(f"Gallery size: {len(gallery_coords):,} GPS points")
print(f"Lat range: [{gallery_coords[:, 0].min():.2f}, {gallery_coords[:, 0].max():.2f}]")
print(f"Lon range: [{gallery_coords[:, 1].min():.2f}, {gallery_coords[:, 1].max():.2f}]")

baseline.build_gallery(gallery_coords)
print("Gallery embeddings computed (with fine-tuned location encoder).")

## 4. Load Query Data

In [ ]:
image_paths, true_coords, landmark_ids = load_query_data(DATA_ROOT)
print(f"Query landmarks: {len(image_paths)}")
print(image_paths[0])

missing = [p for p in image_paths if not p.exists()]
if missing:
    print(f"WARNING: {len(missing)} images not found. First missing: {missing[0]}")
else:
    print("All query images found.")

## 5. Run Inference

In [ ]:
t0 = time.time()
pred_coords = baseline.predict_batch(
    image_paths, batch_size=cfg["inference"]["batch_size"]
)

elapsed = time.time() - t0
print(f"Inference complete: {len(pred_coords)} predictions in {elapsed:.1f}s")

## 6. Evaluate

In [ ]:
pred_lat, pred_lon = pred_coords[:, 0], pred_coords[:, 1]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

thresholds = cfg["evaluation"]["thresholds_km"]
results = accuracy_at_thresholds(pred_lat, pred_lon, true_lat, true_lon, thresholds)
med_err = median_error(pred_lat, pred_lon, true_lat, true_lon)
distances = haversine(pred_lat, pred_lon, true_lat, true_lon)

results_df = pd.DataFrame(
    [{"Threshold (km)": t, "Accuracy (%)": f"{acc * 100:.2f}"} for t, acc in results.items()]
)
print(results_df.to_string(index=False))
print(f"\nMedian error: {med_err:.1f} km")
print(f"Mean error: {distances.mean():.1f} km")

## 7. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.bar([str(t) for t in thresholds], [results[t] * 100 for t in thresholds], color="seagreen")
ax.set_xlabel("Threshold (km)")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy @ k km (fine-tuned)")
ax.set_ylim(0, 100)
for i, t in enumerate(thresholds):
    ax.text(i, results[t] * 100 + 1.5, f"{results[t]*100:.1f}%", ha="center", fontsize=8)

ax = axes[1]
sorted_dist = np.sort(distances)
cdf = np.arange(1, len(sorted_dist) + 1) / len(sorted_dist)
ax.plot(sorted_dist, cdf * 100, color="seagreen", linewidth=1.5)
ax.set_xscale("log")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Cumulative %")
ax.set_title("CDF of Prediction Distances")
ax.axvline(25, color="gray", linestyle="--", alpha=0.5, label="25 km")
ax.axvline(200, color="gray", linestyle=":", alpha=0.5, label="200 km")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.scatter(true_lon, true_lat, s=3, alpha=0.4, label="True", color="steelblue")
ax.scatter(pred_lon, pred_lat, s=3, alpha=0.4, label="Predicted", color="seagreen")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("True vs Predicted GPS (fine-tuned)")
ax.legend(fontsize=7, markerscale=3)

plt.tight_layout()
plt.show()

## 8. Summary: Fine-Tuned vs Zero-Shot

Zero-shot numbers below are copied from `03_geoclip_zeroshot.ipynb` for direct comparison.
Fine-tuned numbers will populate after this notebook runs on HPC with a checkpoint present.

| Threshold (km) | Zero-shot (%) | Fine-tuned (%) |
|---------------:|--------------:|---------------:|
| 1              | 11.30         | _TBD_          |
| 25             | 23.40         | _TBD_          |
| 200            | 43.10         | _TBD_          |
| 750            | 72.40         | _TBD_          |
| 2500           | 93.90         | _TBD_          |

- **Zero-shot median / mean error:** 279.8 km / 626.4 km
- **Fine-tuned median / mean error:** _TBD_

Fine-tuning adapts the Location Encoder and linear image head to the US-centric MMlandmarks
distribution. We expect the largest lift at tight thresholds (1 km / 25 km), where the
pretrained global model performs worst.